In [9]:
import numpy as np
import librosa as lb
import HMMOLTW
from pathlib import Path
import numpy as np
import importlib
import IterativeTrainHMM
import matplotlib.pyplot as plt
import Constants
from scipy.special import logsumexp

importlib.reload(IterativeTrainHMM)
importlib.reload(HMMOLTW)

<module 'HMMOLTW' from 'c:\\Users\\morgp\\Documents\\.HMC\\.Year 3\\E207\\Python\\Project\\E207DTWHMM\\HMMOLTW.py'>

In [10]:
A = np.random.rand(5,5)
print(A)
B = np.max(A, axis = 0)
print(B)

[[0.01295626 0.12497854 0.00624591 0.04559243 0.13922269]
 [0.9930107  0.65112751 0.05815662 0.87998941 0.27968225]
 [0.37557878 0.49327016 0.31181105 0.67181649 0.93670573]
 [0.41799772 0.80480012 0.86934025 0.07134181 0.31146779]
 [0.83190715 0.42148259 0.75219331 0.89716115 0.98536778]]
[0.9930107  0.80480012 0.86934025 0.89716115 0.98536778]


In [11]:
PIECE_ID = "Chopin_Op030No2"
DATA_DIR = Path("data/wav_22050_mono") / PIECE_ID
TRAIN_FRACTION = 0.70
TEST_RECORDING_COUNT = 3

recording_paths = sorted(DATA_DIR.glob("*.wav"))
reference_path = recording_paths[0]

rng = np.random.default_rng(207)
query_pool = np.array(recording_paths[1:], dtype=object)
shuffled_pool = query_pool[rng.permutation(len(query_pool))]
train_count = int(np.ceil(TRAIN_FRACTION * len(recording_paths))) - 1
train_paths = sorted(shuffled_pool[:train_count])
heldout_paths = sorted(shuffled_pool[train_count:])
test_paths = heldout_paths[:TEST_RECORDING_COUNT]

print(f"reference: {reference_path.name}")
print(f"total recordings: {len(recording_paths)}")
print(f"training recordings: {1 + len(train_paths)} including the reference ({(1 + len(train_paths)) / len(recording_paths):.1%})")
print(f"held-out recordings: {len(heldout_paths)}")
print("test queries:")
for path in test_paths:
    print(f"  {path.name}")

reference: Chopin_Op030No2_Ashkenazy-1981_pid9058-19.wav
total recordings: 34
training recordings: 24 including the reference (70.6%)
held-out recordings: 10
test queries:
  Chopin_Op030No2_Chiu-1999_pid9048-19.wav
  Chopin_Op030No2_Cortot-1951_pid9066-19.wav
  Chopin_Op030No2_Fiorentino-1961_pid9065-14.wav


In [12]:
C = np.copy(B)
C[-1] = -np.inf
np.isfinite(C)

array([ True,  True,  True,  True, False])

In [13]:
TestFrom = np.array([1, 2, 3])
TestTo = np.array([2, 3, 4])

D = np.ix_(TestFrom, TestTo)
print(A)
print(A[D])

[[0.01295626 0.12497854 0.00624591 0.04559243 0.13922269]
 [0.9930107  0.65112751 0.05815662 0.87998941 0.27968225]
 [0.37557878 0.49327016 0.31181105 0.67181649 0.93670573]
 [0.41799772 0.80480012 0.86934025 0.07134181 0.31146779]
 [0.83190715 0.42148259 0.75219331 0.89716115 0.98536778]]
[[0.05815662 0.87998941 0.27968225]
 [0.31181105 0.67181649 0.93670573]
 [0.86934025 0.07134181 0.31146779]]


In [14]:
print(B)
print(A)

B[:, np.newaxis] + A

[0.9930107  0.80480012 0.86934025 0.89716115 0.98536778]
[[0.01295626 0.12497854 0.00624591 0.04559243 0.13922269]
 [0.9930107  0.65112751 0.05815662 0.87998941 0.27968225]
 [0.37557878 0.49327016 0.31181105 0.67181649 0.93670573]
 [0.41799772 0.80480012 0.86934025 0.07134181 0.31146779]
 [0.83190715 0.42148259 0.75219331 0.89716115 0.98536778]]


array([[1.00596695, 1.11798924, 0.99925661, 1.03860313, 1.13223339],
       [1.79781082, 1.45592763, 0.86295674, 1.68478952, 1.08448237],
       [1.24491903, 1.3626104 , 1.18115129, 1.54115673, 1.80604597],
       [1.31515887, 1.70196127, 1.7665014 , 0.96850297, 1.20862895],
       [1.81727492, 1.40685037, 1.73756109, 1.88252893, 1.97073555]])

In [15]:
E = np.full(5, -np.inf)
E[3] = 4

print(logsumexp(E))

4.0


In [16]:
QueryFrameSequence =     np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2], dtype=np.int32)
ReferenceFrameSequence = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=np.int32)

QueryIncrements = np.diff(QueryFrameSequence) == 1
ToStates = ReferenceFrameSequence[1:][QueryIncrements]
FromStates = np.roll(ToStates, 1)
FromStates[0] = ReferenceFrameSequence[0]
# FromStates = ReferenceFrameSequence[:-1][QueryIncrements]

MaxState = max(ReferenceFrameSequence + 1)
A = np.zeros((MaxState, MaxState))

print(QueryFrameSequence)
print(ReferenceFrameSequence)

print(FromStates)
print(ToStates)

np.add.at(A, (FromStates, ToStates), 1)
print(A)


[0 0 0 0 1 1 1 1 2 2 2]
[ 0  1  2  3  4  5  6  7  8  9 10]
[0 4]
[4 8]
[[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [18]:
PossiblePreviousStates = [3, 4, 5]
CurrentPossibleStates= [1, 2, 3]

print(A)
print(A[np.ix_(PossiblePreviousStates, CurrentPossibleStates)])

[[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
